# Kaggle Music Generator — ACE-Step 1.5 worker
This notebook installs ACE-Step, reads GITHUB_TOKEN from Kaggle Notebook Secrets, loads the model once, then polls the GitHub Issues queue.

Before running, add a Kaggle Notebook Secret named GITHUB_TOKEN. The GitHub Actions secrets KAGGLE_USER_NAME and KAGGLE_KEY are only used to deploy this notebook.

In [ ]:
import os, sys, subprocess
from pathlib import Path
print('Notebook Python:', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12'], check=True)
print('Python 3.12 is ready for ACE-Step.')

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
os.environ['GITHUB_REPOSITORY'] = 'unisocsu/kaggle-music-generator'
print('GitHub token loaded from Kaggle Secrets (value hidden).')

In [ ]:
repo_dir = Path('/kaggle/working/kaggle-music-generator')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/unisocsu/kaggle-music-generator.git', str(repo_dir)], check=True)
ace_dir = Path('/kaggle/working/ACE-Step-1.5')
if not ace_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ace-step/ACE-Step-1.5.git', str(ace_dir)], check=True)
os.environ['ACESTEP_ROOT'] = str(ace_dir)
os.environ.setdefault('ACESTEP_CONFIG_PATH', 'acestep-v15-turbo')
os.environ.setdefault('ACESTEP_LM_MODEL_PATH', 'acestep-5Hz-lm-0.6B')
os.environ.setdefault('ACESTEP_LM_BACKEND', 'pt')
print('Repositories ready.')

In [ ]:
# Install the official dependency set with Python 3.12.
subprocess.run(['uv', 'sync', '--python', '3.12', '--no-dev'], cwd=str(ace_dir), check=True)
print('ACE-Step dependencies installed.')

In [ ]:
# This loads the model once, then keeps polling until the Kaggle session ends.
worker = repo_dir / 'kaggle' / 'worker.py'
subprocess.run(['uv', 'run', '--python', '3.12', 'python', str(worker)], cwd=str(ace_dir), env=os.environ.copy(), check=True)